# 📐 Fundamentos: Análisis de Componentes Principales (PCA)

## La idea, en una frase
> **PCA busca las direcciones en las que tus datos más varían y las usa como nuevos ejes.** Si te quedas sólo con las primeras, describes casi todo con muchas menos columnas.

### 🔦 La analogía de la sombra
Ilumina una taza con una lámpara: su **sombra** en la pared es una versión 2D de un objeto 3D. Si la lámpara está justo arriba, la sombra es un círculo y ya no sabes que tiene asa. Si la lámpara está de lado, la sombra conserva la forma de la taza.

PCA hace exactamente eso con tus datos: busca **dónde poner la lámpara** para que la sombra (la proyección) quede lo más **extendida** posible, es decir, que conserve la mayor **varianza**. Más varianza = más diferencias entre los puntos = más información.

## 🧭 Contenido
| # | Tema |
|---|---|
| 1 | La sombra que conserva más varianza |
| 2 | PCA a mano en 2D: covarianza → `eigh` → ordenar → proyectar → reconstruir |
| 3 | Lo mismo con scikit-learn (y por qué los signos pueden cambiar) |
| 4 | De 3D a 2D: una espiral aplanada |
| 5 | ¿Por qué estandarizar antes de PCA? |
| 6 | ¿Cuántos componentes? Scree plot, varianza acumulada, codo y Kaiser |
| 7 | Cargas y biplot: ¿qué significa cada componente? |
| 8 | Error de reconstrucción según el número de componentes |
| 9 | ⚠️ Errores comunes |

Hay **3 mini-ejercicios ✍️** en el camino.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_metricas_k(tabla)` | Silhouette, Davies-Bouldin y Calinski-Harabasz según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_metricas_k(tabla):
    """Silhouette, Davies-Bouldin y Calinski-Harabasz para cada k (tabla de tabla_metricas_k)."""
    metricas = [
        ('silhouette', 'Silhouette (↑ mejor)'),
        ('davies_bouldin', 'Davies-Bouldin (↓ mejor)'),
        ('calinski_harabasz', 'Calinski-Harabasz (↑ mejor)'),
    ]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in metricas])
    for i, (col, _) in enumerate(metricas, start=1):
        fig.add_trace(go.Scatter(x=tabla['k'], y=tabla[col], mode='lines+markers', name=col), row=1, col=i)
        fig.update_xaxes(title_text='k', dtick=1, row=1, col=i)
    fig.update_layout(width=1150, height=400, showlegend=False, title='Métricas internas según k')
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

---
# 1️⃣ La sombra que conserva más varianza

Empezamos con **10 puntos en 2D** (dos variables muy relacionadas). Queremos describir cada punto con **un solo número**: su posición sobre una recta que pasa por el centro de la nube. ¿Qué recta elegimos?

Para cada ángulo posible calculamos la **varianza de la sombra** (la proyección de los puntos sobre esa recta).

In [ ]:
# 10 puntos en 2D con correlación positiva
X = np.array([
    [2.5, 2.4], [0.5, 0.7], [2.2, 2.9], [1.9, 2.2], [3.1, 3.0],
    [2.3, 2.7], [2.0, 1.6], [1.0, 1.1], [1.5, 1.6], [1.1, 0.9]
])
X_centrado = X - X.mean(axis=0)


def varianza_de_la_sombra(X_centrado, angulo):
    """Varianza de los puntos proyectados sobre una recta con ese ángulo (en grados)."""
    direccion = np.array([np.cos(np.radians(angulo)), np.sin(np.radians(angulo))])
    sombra = X_centrado @ direccion
    return sombra.var(ddof=1)


angulos = np.arange(0, 180, 1)
varianzas = [varianza_de_la_sombra(X_centrado, a) for a in angulos]

mejor = angulos[np.argmax(varianzas)]
peor = angulos[np.argmin(varianzas)]
print(f"Varianza total de los datos: {X_centrado.var(axis=0, ddof=1).sum():.3f}")
print(f"Mejor ángulo: {mejor}° → varianza de la sombra = {max(varianzas):.3f}")
print(f"Peor ángulo:  {peor}° → varianza de la sombra = {min(varianzas):.3f}")

fig = px.line(
    x=angulos,
    y=varianzas,
    labels={'x': 'Ángulo de la recta (grados)', 'y': 'Varianza de la sombra'},
    title='¿Qué tan extendida queda la sombra según el ángulo?'
)
fig.add_vline(x=mejor, line_dash='dash', line_color='green', annotation_text='máxima')
fig.add_vline(x=peor, line_dash='dash', line_color='red', annotation_text='mínima')
fig.update_layout(width=750, height=400)
fig.show()

In [ ]:
# Dibujamos las dos sombras extremas
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
media = X.mean(axis=0)
for ax, angulo, nombre in [(axes[0], mejor, 'Mejor recta'), (axes[1], peor, 'Peor recta')]:
    u = np.array([np.cos(np.radians(angulo)), np.sin(np.radians(angulo))])
    sombra = X_centrado @ u
    proyeccion = np.outer(sombra, u) + media
    linea = np.array([media - 2.2 * u, media + 2.2 * u])
    ax.plot(linea[:, 0], linea[:, 1], color='gray', lw=1)
    for p, q in zip(X, proyeccion):
        ax.plot([p[0], q[0]], [p[1], q[1]], 'k:', lw=1)
    ax.scatter(X[:, 0], X[:, 1], s=60, color='steelblue', label='Datos')
    ax.scatter(proyeccion[:, 0], proyeccion[:, 1], s=60, color='tomato', label='Sombra')
    ax.set_title(f'{nombre} ({angulo}°): varianza = {sombra.var(ddof=1):.3f}')
    ax.set_aspect('equal')
    ax.set_xlim(-0.5, 4)
    ax.set_ylim(-0.5, 4)
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

✅ **Qué observar:**
- La varianza total de los datos es **1.333**. La mejor recta (**47°**) conserva **1.284** en su sombra: casi todo
- La peor recta (**137°**) sólo conserva **0.049**: los puntos rojos quedan amontonados y ya no se distinguen entre sí
- La mejor recta es justo la que pasa **"a lo largo" de la nube**. Esa recta es el **primer componente principal (PC1)**
- 💡 Las líneas punteadas son lo que **perdemos** al quedarnos con la sombra. Maximizar la varianza de la sombra es lo mismo que **minimizar** esas distancias

Buscar el ángulo a fuerza bruta funciona en 2D, pero no con 13 o 784 variables. Para eso está el álgebra lineal.

---
# 2️⃣ PCA a mano en 2D

La receta completa tiene 5 pasos:

| Paso | Qué hacemos | En símbolos |
|---|---|---|
| 1 | **Centrar**: restar la media de cada columna | $X_c = X - \mu$ |
| 2 | **Covarianza**: cómo varían las columnas juntas | $C = \frac{1}{n-1} X_c^\top X_c$ |
| 3 | **Eigenvectores y eigenvalores** de $C$ | $C\,v = \lambda\,v$ |
| 4 | **Ordenar** de mayor a menor $\lambda$ | $\lambda_1 \ge \lambda_2 \ge \dots$ |
| 5 | **Proyectar** (y, si queremos, **reconstruir**) | $z = X_c\,v_1$, $\hat{X} = z\,v_1^\top + \mu$ |

- Cada **eigenvector** $v$ es una **dirección** (un componente principal)
- Su **eigenvalor** $\lambda$ es la **varianza de la sombra** en esa dirección

In [ ]:
# Paso 1: centrar
media = X.mean(axis=0)
X_centrado = X - media

# Paso 2: matriz de covarianza (2 x 2)
cov = np.cov(X_centrado.T)
print("Matriz de covarianza:")
print(cov.round(4))

En la diagonal están las varianzas de cada variable (0.617 y 0.717) y fuera de ella la covarianza (0.615): positiva y grande, las dos variables suben juntas.

Para el paso 3 usamos **`np.linalg.eigh`** y no `np.linalg.eig`:
- La covarianza es **simétrica**; `eigh` está hecha para matrices simétricas: garantiza eigenvalores **reales** y eigenvectores **perpendiculares** (ortonormales)
- `eig` es para cualquier matriz y, por redondeo, puede regresar números complejos con parte imaginaria 0
- ⚠️ `eigh` regresa los eigenvalores de **menor a mayor**: por eso existe el paso 4

In [ ]:
# Paso 3: eigenvalores y eigenvectores (eigh: para matrices simétricas)
eigenvalores, eigenvectores = np.linalg.eigh(cov)
print("Tal como los regresa eigh (de menor a mayor):", eigenvalores.round(4))

# Paso 4: ordenar de mayor a menor varianza
orden = np.argsort(eigenvalores)[::-1]
eigenvalores = eigenvalores[orden]
eigenvectores = eigenvectores[:, orden]   # cada COLUMNA es un componente

print("Ordenados:", eigenvalores.round(4))
print("Varianza explicada:", (eigenvalores / eigenvalores.sum()).round(4))
print("PC1 =", eigenvectores[:, 0].round(4))
print("PC2 =", eigenvectores[:, 1].round(4))

✅ **Qué observar:**
- $\lambda_1 = 1.284$ y $\lambda_2 = 0.049$: exactamente las varianzas de la mejor y la peor sombra del punto 1
- PC1 explica el **96.3%** de la varianza; PC2 sólo el **3.7%**
- PC1 = (0.678, 0.735): apunta a **47°**, la misma recta que encontramos a fuerza bruta. PC2 es perpendicular a PC1

In [ ]:
# Paso 5: proyectar sobre PC1 (de 2 columnas a 1) y reconstruir
v1 = eigenvectores[:, 0]
z = X_centrado @ v1                       # un número por punto
X_reconstruido = np.outer(z, v1) + media  # de vuelta a 2D, pero sobre la recta de PC1

error = ((X - X_reconstruido) ** 2).sum()
print("Coordenada en PC1 de los primeros 3 puntos:", z[:3].round(3))
print(f"Error de reconstrucción (suma de distancias²): {error:.3f}")
print(f"λ2 × (n - 1) = {eigenvalores[1] * (len(X) - 1):.3f}")

# Gráfica: datos, componentes y líneas de error
fig, ax = plt.subplots(figsize=(7, 6))
for p, q in zip(X, X_reconstruido):
    ax.plot([p[0], q[0]], [p[1], q[1]], 'k:', lw=1)
ax.scatter(X[:, 0], X[:, 1], s=60, color='steelblue', label='Datos originales')
ax.scatter(X_reconstruido[:, 0], X_reconstruido[:, 1], s=60, color='tomato', label='Reconstruidos con PC1')
for v, lam, color, nombre in [(eigenvectores[:, 0], eigenvalores[0], 'green', 'PC1'),
                              (eigenvectores[:, 1], eigenvalores[1], 'purple', 'PC2')]:
    largo = 1.5 * np.sqrt(lam)  # flecha proporcional a la desviación estándar
    ax.annotate('', xy=media + largo * v, xytext=media,
                arrowprops={'arrowstyle': '->', 'color': color, 'lw': 2.5})
    ax.text(*(media + largo * v * 1.15), nombre, color=color, fontsize=12, weight='bold')
ax.set_aspect('equal')
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_title('PCA a mano: proyección sobre PC1 y error de reconstrucción')
ax.grid(alpha=0.3)
ax.legend(loc='upper left')
plt.show()

✅ **Qué observar:**
- Cada punto ahora se describe con **un solo número** ($z$): su posición a lo largo de PC1
- Los puntos rojos son la **reconstrucción**: todos caen sobre la recta de PC1. Las líneas punteadas son el **error**
- 💡 El error total (**0.442**) es exactamente $\lambda_2 \times (n-1)$: **lo que perdemos es la varianza de los componentes que tiramos**

### ✍️ Mini-ejercicio 1: la peor lámpara

Repite el paso 5, pero proyecta sobre **PC2** en lugar de PC1. ¿Cuánto vale ahora el error de reconstrucción? ¿Coincide con $\lambda_1 \times (n-1)$?

In [ ]:
# 1. v2 = eigenvectores[:, 1]
# 2. z2 = X_centrado @ v2
# 3. X_reconstruido_pc2 = np.outer(z2, v2) + media
# 4. error_pc2 = ((X - X_reconstruido_pc2) ** 2).sum()  → imprímelo junto a eigenvalores[0] * (len(X) - 1)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Cuántas veces más grande es el error con PC2 que con PC1? ¿Por qué PCA ordena los componentes de mayor a menor varianza?

RELLENAR CON TUS COMENTARIOS

---
# 3️⃣ Lo mismo con scikit-learn

En la práctica nadie calcula los eigenvectores a mano: `sklearn.decomposition.PCA` hace los 5 pasos (centra por dentro, pero **no escala**: eso es tarea tuya).

| Atributo | Qué es | A mano |
|---|---|---|
| `components_` | Una **fila** por componente | columnas de `eigenvectores` |
| `explained_variance_` | Varianza de cada componente | `eigenvalores` |
| `explained_variance_ratio_` | Proporción de la varianza total | `eigenvalores / eigenvalores.sum()` |
| `transform(X)` | Coordenadas en los componentes (*scores*) | `X_centrado @ eigenvectores` |
| `inverse_transform(Z)` | Regresa al espacio original | `Z @ eigenvectores.T + media` |

In [ ]:
pca_2d = PCA(n_components=2)
scores_2d = pca_2d.fit_transform(X)

print("sklearn components_ (filas):")
print(pca_2d.components_.round(4))
print("A mano (eigenvectores transpuestos):")
print(eigenvectores.T.round(4))
print()
print("Varianzas sklearn:", pca_2d.explained_variance_.round(4), "| a mano:", eigenvalores.round(4))
print("¿Mismos componentes salvo el signo?", np.allclose(np.abs(pca_2d.components_), np.abs(eigenvectores.T)))
print("¿Mismos scores salvo el signo?   ", np.allclose(np.abs(scores_2d), np.abs(X_centrado @ eigenvectores)))

✅ **Qué observar:**
- Las varianzas son idénticas (**1.284** y **0.049**) y PC1 coincide
- PC2 sale con el **signo invertido**: sklearn da (0.735, −0.678) y nosotros (−0.735, 0.678)

⚠️ **¿Por qué cambia el signo?** Si $v$ es eigenvector, $-v$ también lo es: $C(-v) = \lambda(-v)$. Ambos describen **la misma recta**; sólo cambia hacia dónde apunta la flecha. Cada librería (o versión) elige un sentido por convención.

Consecuencias prácticas:
- Los *scores* de ese componente se multiplican por −1 (el dibujo se refleja como en un espejo), pero las distancias entre puntos no cambian
- Al interpretar, lo que importa son los **signos relativos** dentro de un componente ("estas variables suben juntas y aquella baja"), no el signo absoluto

---
# 4️⃣ De 3D a 2D: una espiral aplanada

Generamos 200 puntos en 3D que viven **casi** sobre un plano inclinado (la tercera coordenada es casi una combinación de las otras dos más un poco de ruido). PCA debería encontrar ese plano.

In [ ]:
rng = np.random.default_rng(42)
m = 200
angulo = rng.uniform(-0.5, 3 * np.pi / 2 - 0.5, m)
X3 = np.empty((m, 3))
X3[:, 0] = np.cos(angulo) + np.sin(angulo) / 2 + rng.normal(0, 0.05, m)
X3[:, 1] = 0.7 * np.sin(angulo) + rng.normal(0, 0.05, m)
X3[:, 2] = 0.1 * X3[:, 0] + 0.3 * X3[:, 1] + rng.normal(0, 0.1, m)

pca_espiral = PCA(n_components=2)
X2 = pca_espiral.fit_transform(X3)
X3_reconstruido = pca_espiral.inverse_transform(X2)

ratio = pca_espiral.explained_variance_ratio_
print(f"Varianza explicada por PC1 y PC2: {ratio.round(4)}")
print(f"Total conservado: {ratio.sum():.2%} | perdido: {1 - ratio.sum():.2%}")
print(f"Distancia media entre cada punto y su reconstrucción: {np.linalg.norm(X3 - X3_reconstruido, axis=1).mean():.4f}")

In [ ]:
# El plano que encontró PCA, dibujado en 3D
g1, g2 = np.meshgrid(np.linspace(X2[:, 0].min(), X2[:, 0].max(), 10),
                     np.linspace(X2[:, 1].min(), X2[:, 1].max(), 10))
plano = pca_espiral.inverse_transform(np.column_stack([g1.ravel(), g2.ravel()]))

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=X3[:, 0], y=X3[:, 1], z=X3[:, 2], mode='markers',
                           marker={'size': 3, 'color': angulo, 'colorscale': 'Viridis'}, name='Datos 3D'))
fig.add_trace(go.Mesh3d(x=plano[:, 0], y=plano[:, 1], z=plano[:, 2], opacity=0.3, color='gray',
                        name='Plano PC1-PC2'))
fig.update_layout(title='Datos 3D y el plano de PC1-PC2 (gíralo con el mouse)', width=800, height=600)
fig.show()

fig = px.scatter(x=X2[:, 0], y=X2[:, 1], color=angulo, color_continuous_scale='Viridis',
                 labels={'x': 'PC1', 'y': 'PC2', 'color': 'ángulo'},
                 title='La misma espiral, aplanada en 2D')
fig.update_yaxes(scaleanchor='x')
fig.update_layout(width=750, height=500)
fig.show()

✅ **Qué observar:**
- PC1 conserva el **86.0%** de la varianza y PC2 el **13.0%**: con 2 columnas guardamos el **98.99%**; sólo perdemos **1.01%**
- En promedio, cada punto queda a **0.078** unidades de su reconstrucción: casi nada, porque los datos ya vivían cerca de un plano
- La espiral se ve completa en 2D (los colores siguen el mismo orden). PCA encontró la "pared" correcta para la sombra

---
# 5️⃣ ¿Por qué estandarizar antes de PCA?

PCA busca **varianza**, y la varianza depende de las **unidades**. Una variable medida en pesos tiene varianzas de millones; una medida en metros, de décimas. Sin escalar, PCA no encuentra "patrones": encuentra **la variable con los números más grandes**.

Lo vemos con 200 personas: `altura_cm` y `peso_kg` están muy relacionadas; `ingreso_mensual` (en pesos) es independiente de ambas.

In [ ]:
rng = np.random.default_rng(42)
n = 200
altura = rng.normal(170, 10, n)
peso = 0.9 * (altura - 170) + rng.normal(70, 6, n)
ingreso = rng.normal(25_000, 8_000, n)
personas = pd.DataFrame({'altura_cm': altura, 'peso_kg': peso, 'ingreso_mensual': ingreso})
columnas_personas = list(personas.columns)

print("Desviación estándar de cada variable:")
print(personas.std().round(1))
print(f"\nLa varianza del ingreso es {personas['ingreso_mensual'].var() / personas['altura_cm'].var():,.0f} veces la de la altura")
print(f"Correlación altura-peso: {personas['altura_cm'].corr(personas['peso_kg']):.2f}")

In [ ]:
# Sin escalar
pca_sin_escalar = PCA().fit(personas)
# Escalando (media 0, desviación estándar 1)
personas_scaled = StandardScaler().fit_transform(personas)
pca_escalado = PCA().fit(personas_scaled)

for nombre, modelo in [('SIN escalar', pca_sin_escalar), ('ESCALADO', pca_escalado)]:
    print(f"--- {nombre} ---")
    print("Varianza explicada:", modelo.explained_variance_ratio_.round(3))
    print(pd.DataFrame(modelo.components_[:2], columns=columnas_personas, index=['PC1', 'PC2']).round(3))
    print()

✅ **Qué observar:**
- **Sin escalar**, PC1 explica el **100%** de la varianza y es **sólo** `ingreso_mensual` (carga 1.0): PCA nos devolvió la columna con los números más grandes. La relación altura-peso (correlación **0.78**) quedó escondida
- **Escalado**, PC1 (**59.3%**) combina `altura_cm` y `peso_kg` con pesos casi iguales (0.71): es una medida de **"tamaño corporal"**. El ingreso queda solo en PC2 (**33.3%**)
- 💡 Regla: si tus variables están en **unidades distintas**, estandariza antes de PCA. La única excepción es cuando todas comparten la misma unidad y escala (p. ej. los píxeles de una imagen)

---
# 6️⃣ ¿Cuántos componentes me quedo?

Con 2 o 3 variables es fácil. Con muchas, usamos el **scree plot** (*scree* = las piedras que caen al pie de una montaña): barras con la varianza de cada componente y una línea con la **varianza acumulada**.

Usamos el dataset de **cáncer de mama** de Wisconsin (`load_breast_cancer`): **569 tumores** descritos por **30 medidas** de sus células (radio, textura, perímetro, área, suavidad…, cada una en promedio, error y peor valor).

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)
X_cancer = StandardScaler().fit_transform(cancer.data)
print(f"{X_cancer.shape[0]} tumores × {X_cancer.shape[1]} variables")

pca_cancer = PCA().fit(X_cancer)
plot_varianza_explicada(pca_cancer)

### Tres criterios para decidir

| Criterio | Regla | Cuándo usarlo |
|---|---|---|
| **Varianza acumulada** | Quedarse con los componentes que suman **80% a 95%** | Compresión o preprocesamiento: controlas cuánto pierdes |
| **Codo** | Cortar donde las barras dejan de caer bruscamente | Exploración: los componentes después del codo aportan poco cada uno |
| **Kaiser** | Quedarse con los componentes con eigenvalor **> 1** (sólo con datos **estandarizados**) | Un componente con $\lambda < 1$ explica menos que **una** variable original |

Para visualizar, la respuesta casi siempre es **2** (o 3). Los criterios no tienen que coincidir.

In [ ]:
acumulada = np.cumsum(pca_cancer.explained_variance_ratio_)
for umbral in (0.80, 0.90, 0.95):
    print(f"Componentes para {umbral:.0%}: {np.argmax(acumulada >= umbral) + 1}")
print(f"Kaiser (eigenvalor > 1): {(pca_cancer.explained_variance_ > 1).sum()}")
print(f"PC1 y PC2 juntos: {acumulada[1]:.1%}")

✅ **Qué observar:**
- PC1 sola explica **44.3%** de la varianza de 30 variables: muchas medidas del tumor cuentan la misma historia (el **tamaño**: radio, perímetro y área van juntos)
- Para el **80%** bastan **5** componentes y para el **95%**, **10**. Kaiser sugiere **6**
- El codo está alrededor de **PC3**: después, cada componente aporta menos de 7%
- Con 2 componentes conservamos **63.2%**: suficiente para dibujar, no para comprimir

💡 Atajo: `PCA(n_components=0.95)` elige solo el número de componentes que llega al 95%. El número elegido queda en `pca.n_components_`.

### ✍️ Mini-ejercicio 2

Ajusta un PCA sobre `X_cancer` que conserve el **90%** de la varianza usando `n_components=0.90`. ¿Cuántos componentes eligió? ¿Cuánta varianza conserva exactamente?

In [ ]:
# 1. pca_cancer_90 = PCA(n_components=0.90).fit(X_cancer)
# 2. Imprime pca_cancer_90.n_components_
# 3. Imprime pca_cancer_90.explained_variance_ratio_.sum()
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Cuántos componentes necesitas para el 90%? ¿Qué fracción de las 30 columnas originales es?

RELLENAR CON TUS COMENTARIOS

---
# 7️⃣ Cargas y biplot: ¿qué significa cada componente?

Un componente es una **mezcla** de las variables originales:

$$PC_1 = w_1 \cdot x_1 + w_2 \cdot x_2 + \dots + w_p \cdot x_p$$

Los pesos $w$ se llaman **cargas** (*loadings*) y son las coordenadas del eigenvector (`pca.components_`). Una carga grande en valor absoluto = esa variable pesa mucho en ese componente; el **signo** dice si sube o baja con él.

Usamos **Iris** (150 flores, 4 medidas en cm) porque con 4 variables las cargas se leen fácilmente.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
columnas_iris = ['largo_sepalo', 'ancho_sepalo', 'largo_petalo', 'ancho_petalo']
X_iris = StandardScaler().fit_transform(iris.data)

pca_iris = PCA().fit(X_iris)
scores_iris = pca_iris.transform(X_iris)
print("Varianza explicada:", pca_iris.explained_variance_ratio_.round(3))

plot_cargas(pca_iris, columnas_iris, n_componentes=2)

✅ **Qué observar:**
- **PC1 (73.0%)**: `largo_petalo` (0.58), `ancho_petalo` (0.57) y `largo_sepalo` (0.52) con el mismo signo → mide el **tamaño de la flor**
- `ancho_sepalo` pesa poco en PC1 y con signo contrario (−0.27)
- ⚠️ PC1 **no es** "el largo del pétalo": es una mezcla de 4 medidas. Ponle un nombre que describa la mezcla

El **biplot** junta todo en una gráfica: cada punto es una flor en el plano PC1-PC2 y cada flecha es una variable original (sus cargas en PC1 y PC2).
- Flechas que apuntan **en la misma dirección** = variables correlacionadas
- Flechas **largas** = variables bien representadas en estos dos componentes
- Un punto que está **hacia donde apunta** una flecha tiene valores altos de esa variable

In [ ]:
especie = iris.target_names[iris.target]
plot_biplot(scores_iris, pca_iris, columnas_iris, labels=especie, title='Biplot de Iris')

✅ **Qué observar:**
- Las flechas de `largo_petalo` y `ancho_petalo` casi se enciman: son variables redundantes
- *Setosa* queda sola a la izquierda (flores pequeñas: valores bajos de PC1); *versicolor* y *virginica* se traslapan un poco a la derecha
- Coloreamos por especie para **validar**: en un problema real no sabríamos la especie, y el biplot nos ayudaría a descubrir que existen grupos

### ✍️ Mini-ejercicio 3

Mira las cargas de **PC2** en la gráfica de cargas y en el biplot. ¿Qué variable domina PC2? ¿Qué nombre le pondrías a ese componente?

RELLENAR CON TUS COMENTARIOS

💡 **Pista para verificar:** imprime `pca_iris.components_[1]` y busca la carga con mayor valor absoluto.

---
# 8️⃣ Error de reconstrucción según el número de componentes

Con $k$ componentes comprimimos y luego reconstruimos (`inverse_transform`). Medimos el error cuadrático medio entre los datos (escalados) y su reconstrucción, para $k = 1, \dots, 30$ en el dataset de cáncer.

In [ ]:
filas = []
for k in range(1, X_cancer.shape[1] + 1):
    pca_k = PCA(n_components=k).fit(X_cancer)
    X_rec = pca_k.inverse_transform(pca_k.transform(X_cancer))
    filas.append({
        'k': k,
        'error_reconstruccion': ((X_cancer - X_rec) ** 2).mean(),
        'varianza_no_explicada': 1 - pca_k.explained_variance_ratio_.sum()
    })
errores = pd.DataFrame(filas)

fig = px.line(
    errores,
    x='k',
    y=['error_reconstruccion', 'varianza_no_explicada'],
    markers=True,
    labels={'k': 'Número de componentes (k)', 'value': 'Valor', 'variable': ''},
    title='Error de reconstrucción vs varianza no explicada'
)
fig.update_layout(width=800, height=450)
fig.show()
errores.query('k in [1, 2, 5, 10, 20]').round(3)

✅ **Qué observar:**
- Con **1** componente el error es **0.557**; con **5**, **0.153**; con **10**, **0.048**; con 30 es **0**
- Las dos líneas están **una encima de la otra**: con datos estandarizados, el error de reconstrucción **es** la varianza que no explicamos
- 💡 Elegir "quiero conservar el 95% de la varianza" es lo mismo que elegir "acepto perder el 5% de la información al reconstruir"

---
# 9️⃣ ⚠️ Errores comunes

| Error | Qué pasa | Cómo evitarlo |
|---|---|---|
| **No escalar** | PC1 = la variable con la varianza más grande (sección 5) | `StandardScaler` antes de PCA cuando las unidades son distintas |
| **Interpretar un PC como una variable original** | "PC1 es el largo del pétalo" es falso: es una mezcla de todas | Lee las **cargas** y ponle un nombre a la mezcla |
| **Usar PCA en variables categóricas** | Códigos como 1 = norte, 2 = centro, 3 = sur no tienen varianza con sentido; los *dummies* 0/1 tampoco se comportan como variables continuas | Usa PCA con variables numéricas continuas; para categóricas existen técnicas como MCA o FAMD |
| **Olvidar que PCA es lineal** | Sólo encuentra rectas y planos; si los datos viven sobre una curva, la sombra los revuelve | Para estructuras no lineales: t-SNE, UMAP o Kernel PCA |
| **Comparar signos entre corridas** | El signo de un componente es arbitrario (sección 3) | Interpreta signos **relativos** dentro de un componente |
| **Ajustar el PCA sobre unos datos y transformar otros sin pensarlo** | Las direcciones cambian si cambian los datos | `fit` sobre los datos que vas a analizar; si hay modelo supervisado después, `fit` sólo en train |

Veamos el de la linealidad con dos círculos concéntricos: ninguna recta los separa.

In [ ]:
from sklearn.datasets import make_circles

X_circ, y_circ = make_circles(n_samples=400, factor=0.4, noise=0.05, random_state=42)
pc1_circ = PCA(n_components=1).fit_transform(X_circ).ravel()

grupo = np.where(y_circ == 1, 'círculo interior', 'círculo exterior')
fig = make_subplots(rows=1, cols=2, subplot_titles=['Datos en 2D', 'Sombra sobre PC1 (1D)'])
for g, color in [('círculo exterior', 'steelblue'), ('círculo interior', 'tomato')]:
    m = grupo == g
    fig.add_trace(go.Scatter(x=X_circ[m, 0], y=X_circ[m, 1], mode='markers', name=g,
                             marker={'color': color, 'size': 5}), row=1, col=1)
    fig.add_trace(go.Histogram(x=pc1_circ[m], name=g, marker={'color': color}, opacity=0.6,
                               showlegend=False), row=1, col=2)
fig.update_layout(barmode='overlay', width=950, height=420, title='PCA es lineal: no puede "desenrollar" círculos')
fig.show()

✅ **Qué observar:** en la sombra sobre PC1 los dos círculos quedan **revueltos**: el interior cae justo en medio del exterior. No es un error de PCA; es su naturaleza lineal. Para estos casos existen métodos no lineales (t-SNE y UMAP, muy usados para visualizar) que se salen del alcance de esta sesión.

---
## 📝 Resumen

1. PCA encuentra las direcciones de **máxima varianza**: la "sombra" que conserva más información
2. Receta: **centrar → covarianza → `eigh` → ordenar → proyectar**. El eigenvalor es la varianza de cada componente
3. Lo que se pierde al reconstruir es la varianza de los componentes descartados
4. **Estandariza** cuando las unidades son distintas; si no, PC1 es la variable más grande
5. Elige cuántos componentes con la **varianza acumulada** (80-95%), el **codo** y **Kaiser**, según el objetivo
6. Interpreta con **cargas** y **biplot**; un PC es una mezcla, no una variable original

En el laboratorio aplicarás todo esto a 13 medidas químicas de vinos.